In [20]:
from typing import List,Optional

from utils.evaluate_rag import *
from utils.helper_functions import *
import os
import sys
from dotenv import load_dotenv
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi

load_dotenv(dotenv_path="/Users/nilasark/advanced/.env")

path="/Users/nilasark/advanced/data/hyde_rag.pdf"

In [4]:
def encode_pdf_and_split_documents(path,chunk_size:int|None=None,chunk_overlap:int|None=None):
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
    chunker=RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    loader=PyPDFLoader(file_path="/Users/nilasark/advanced/data/hyde_rag.pdf")
    loaded_documents=loader.load()
    chunks=chunker.split_documents(loaded_documents)
    cleaned_text=replace_tab_with_space(chunks)
    vectorstore=Chroma.from_documents(cleaned_text,embeddings)

    return vectorstore,cleaned_text


In [5]:
vectorstore,clened_text=encode_pdf_and_split_documents(path)

Create BM25 index to retrieve document by keyword

In [16]:
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?"
]

tokenized_corpus=[doc.split(" ") for doc in corpus]
bm25=BM25Okapi(tokenized_corpus)

query="windy London"
tokenized_query=query.split(" ")
bm25_scores=bm25.get_scores(tokenized_query)
doc_top_n=bm25.get_top_n(tokenized_query,corpus,n=2)
print(doc_top_n)

['It is quite windy in London', 'How is the weather today?']


In [24]:

def create_bm25_index(documents:List[Document])->BM25Okapi:
    tokenized_documents=[doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_documents)


In [25]:
bm25=create_bm25_index(clened_text)